# الدرس 08 - نمط تصميم الوكلاء المتعددين


## الإعداد


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## لماذا أنظمة الوكلاء المتعددين؟

المهام الواقعية مثل تخطيط الرحلات تتطلب العديد من أنواع الخبرات المختلفة — اللوجستيات، المعرفة المحلية، الميزانية، والمزيد. يصبح من الصعب على وكيل واحد محاولة التعامل مع كل شيء بسرعة.

تحل أنظمة الوكلاء المتعددين هذه المشكلة من خلال **التخصص**: يركز كل وكيل على مجال واحد من الخبرة، مما ينتج عنه نتائج ذات جودة أعلى مقارنة بالعامة. كما أنها تحسن **قابلية التوسع** — يمكنك إضافة وكلاء جدد (مثل أخصائي الرحلات الجوية، ناقد المطاعم) دون الحاجة إلى إعادة كتابة سير العمل الموجود. تتكامل الوكلاء معًا من خلال خط أنابيب منظم، يمرر السياق من واحد للآخر.


## إنشاء وكلاء متخصصين


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## بناء سير عمل متسلسل

يتيح لك `WorkflowBuilder` توصيل الوكلاء في رسم بياني موجه. هنا نقوم بإنشاء خط أنابيب بسيط من خطوتين: يقوم **مخطط السفر** بوضع مسار الرحلة، ثم يقوم **موظف خدمة السفر** بمراجعته وتحسينه.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## إضافة مزيد من الوكلاء إلى سير العمل

واحدة من أكبر مزايا نمط الوكلاء المتعددين هي سهولة التوسيع. أدناه نضيف وكيل **BudgetReviewer** الذي يتحقق من الخطة مقابل ميزانية المسافر، ويشير إلى العناصر التي قد تدفع التكاليف إلى ما فوق الحد، ويقترح بدائل لتوفير المال. يعمل سير العمل الآن بثلاثة وكلاء بالتتابع:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## الملخص

في هذا الدرس تعلمت كيفية:

1. **إنشاء وكلاء متخصصين** — كل واحد له دور محدد (التخطيط، خدمة العملاء، مراجعة الميزانية).
2. **ربط الوكلاء في سير عمل متسلسل** باستخدام `WorkflowBuilder` و `add_edge`.
3. **بث الإخراج** من خط أنابيب متعدد الوكلاء، مع تتبع الوكيل الذي يتحدث.
4. **تمديد سير العمل** بإضافة وكلاء جدد إلى السلسلة دون تعديل الوكلاء الحاليين.

نمط تصميم الوكلاء المتعددين يحافظ على بساطة كل وكيل بينما ينتج نتائج أغنى وأكثر مراجعة بدقة مما قد يحققه أي وكيل منفرد بمفرده.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى لتحقيق الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي باللغة الأصلية هو المصدر الموثوق به. بالنسبة للمعلومات الحساسة، يُنصح بالاستعانة بترجمة بشرية احترافية. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
